## Importing Packages

In [15]:
import pandas as pd
import polars as pl

import os
import glob

from tvDatafeed import TvDatafeed, Interval # pip install tradingview-datafeed

from datetime import date

import warnings
warnings.filterwarnings("ignore", message = "Index.is_numeric is deprecated")

## Creating the Dataset

### Downloading NIFTY 500 data (index constituents downloaded from NSE website on 18th March 2026)

In [16]:
stock_prices_dir = "data/nifty_500_stocks/"
master_files_dir = "data/master_files/"

stock_prices_file = master_files_dir + "nifty_500_stocks.parquet"
index_prices_file = master_files_dir + "nifty_50_index.parquet"

In [17]:
# csv file downloaded from NSE website (on 18th March 2026)
nifty_500_list = pl.read_csv('nifty_500_list.csv')

nifty_500_symbols = nifty_500_list['Symbol'].to_list()

# Correcting the mismatch in the usage of - instead of _ in the two different datasets
nifty_500_symbols = [s.replace("-", "_") for s in nifty_500_symbols]

print(f'Total stocks: {len(nifty_500_symbols)}')

Total stocks: 500


In [18]:
# Initializing TvDatafeed for data
tv = TvDatafeed()

In [19]:
failed = []

# Downlaod only when the master file does not already exist (avoids unnecessary downloads each time the notebook is run)
if not os.path.exists(stock_prices_file):
    for symbol in nifty_500_symbols:
        # Downloading 5000 bars (approx 20 years). Will filter later as per requirement
        try:
            df = tv.get_hist(
                symbol = symbol,
                exchange = 'NSE',
                interval = Interval.in_daily,
                n_bars = 5040
                )

            # Using Parquet files to speed up computation given the size of the dataset
            df.to_parquet(f"{stock_prices_dir}{symbol}.parquet")

        except Exception as e:
            print(f"Retry failed for {symbol}: {e}")
            failed.append(symbol)


    # TVFeed fails to download the data sometimes. This while loop ensures that all 500 stocks are downloaded properly
    while len(failed) != 0:

        for symbol in failed[:]:
            try:
                df = tv.get_hist(
                    symbol = symbol,
                    exchange = 'NSE',
                    interval = Interval.in_daily,
                    n_bars = 5040
                    )

                df.to_parquet(f"{stock_prices_dir}{symbol}.parquet")
                failed.remove(symbol)

            except:
                pass

    print("\nFailed cases were run repeatedly until success. Data download is complete.")

else:
    print("Skipping as master file already exists")

Skipping as master file already exists


### Downloading NIFTY 50 index data for benchmarking and market regime filtering

In [21]:
# Downlaod only when the file does not already exist
if not os.path.exists(index_prices_file):
    # Downloading 5000 bars (approx 20 years). Will filter as per requirement
    index_df = tv.get_hist(
        symbol = 'NIFTY',
        exchange = 'NSE',
        interval = Interval.in_daily,
        n_bars = 5040
    )

    index_df = index_df.reset_index()
    index_df = index_df.rename(columns = {"datetime": "date"})
    index_df.to_parquet(index_prices_file)

else:
    index_df = pd.read_parquet(index_prices_file)
    print("Skipping as NIFTY 50 data already exists")

Skipping as NIFTY 50 data already exists


### Merging individual stock data files into a single master file

In [22]:
def merge_parquet_files():
    # Merging only when the final master file does not already exist
    if os.path.exists(stock_prices_file):
        print("Skipping to avoid overwrite as merged data file already exists")
        return

    files = glob.glob(os.path.join(stock_prices_dir, "*.parquet"))

    if len(files) == 0:
        print("No parquet files found in data/nifty_500_stocks/")
        return

    dfs = []

    for file in files:
        symbol = os.path.basename(file).replace(".parquet", "")
        df = (pl.read_parquet(file).with_columns([pl.lit(symbol).alias("symbol")]))
        df = df.rename({"datetime": "date"})
        dfs.append(df)

    final_df = pl.concat(dfs, how = "vertical")
    final_df = final_df.sort(["symbol", "date"])
    final_df.write_parquet(stock_prices_file)

    print(f"Master file created: {stock_prices_file}")
    print(f"Rows: {final_df.height}, Columns: {final_df.width}")


if __name__ == "__main__":
    merge_parquet_files()

Skipping to avoid overwrite as merged data file already exists
